In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv(
    "best_cells_8d_prepared.csv",
    parse_dates=["date"],
)

predictors = ["lai", "sm", "hu", "tg"]
target = "dTa"

# Assume dates mark the start of eight-day windows.
data["window_end"] = data["date"] + pd.Timedelta(days=7)

train = (
    data.loc[
        (data["date"] >= "2010-01-01")
        & (data["window_end"] < "2011-01-01")
    ]
    .dropna(subset=[target] + predictors)
    .sort_values(["date", "lon", "lat"])
    .copy()
)

test = (
    data.loc[
        (data["date"] >= "2011-01-01")
        & (data["window_end"] < "2012-01-01")
    ]
    .dropna(subset=[target] + predictors)
    .sort_values(["date", "lon", "lat"])
    .copy()
)

counts = pd.concat(
    [
        train.groupby(["lon", "lat"]).size().rename("train"),
        test.groupby(["lon", "lat"]).size().rename("test"),
    ],
    axis=1,
)

display(counts)

assert len(train) == 114
assert len(test) == 119

print("Training observations:", len(train))
print("Test observations:", len(test))

,,train,test
lon,lat,,
-9.125,52.125,38,41
-4.125,41.375,42,43
2.125,45.875,34,35


Training observations: 114
Test observations: 119


In [2]:
validation_periods = [
    ("2010-05-01", "2010-07-01"),
    ("2010-07-01", "2010-09-01"),
    ("2010-09-01", "2011-01-01"),
]

folds = []
fold_count_records = []

for fold_number, (start, end) in enumerate(
    validation_periods, start=1
):
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    fitting = train.loc[
        train["window_end"] < start
    ].copy()

    validation = train.loc[
        (train["date"] >= start)
        & (train["window_end"] < end)
    ].copy()

    assert fitting["window_end"].max() < validation["date"].min()

    folds.append({
        "fold": fold_number,
        "fitting": fitting,
        "validation": validation,
    })

    for (lon, lat), cell in train.groupby(["lon", "lat"]):
        fold_count_records.append({
            "fold": fold_number,
            "lon": lon,
            "lat": lat,
            "n_fit": len(fitting.loc[
                (fitting["lon"] == lon)
                & (fitting["lat"] == lat)
            ]),
            "n_validation": len(validation.loc[
                (validation["lon"] == lon)
                & (validation["lat"] == lat)
            ]),
        })

fold_counts = pd.DataFrame(fold_count_records)
display(fold_counts)

,fold,lon,lat,n_fit,n_validation
0,1,-9.125,52.125,13,7
1,1,-4.125,41.375,14,7
2,1,2.125,45.875,8,7
3,2,-9.125,52.125,20,7
4,2,-4.125,41.375,21,7
5,2,2.125,45.875,15,7
6,3,-9.125,52.125,28,9
7,3,-4.125,41.375,29,12
8,3,2.125,45.875,23,10


In [3]:
def add_features(frame):
    result = frame.copy()
    result["sm_lai"] = result["sm"] * result["lai"]
    result["sm_squared"] = result["sm"] ** 2
    return result


feature_sets = {
    "Original four": predictors,
    "Original + SM×LAI": predictors + ["sm_lai"],
    "Original + SM²": predictors + ["sm_squared"],
}

# Apply the same deterministic calculations to each fold.
for fold in folds:
    fold["fitting"] = add_features(fold["fitting"])
    fold["validation"] = add_features(fold["validation"])

display(
    folds[0]["fitting"][
        ["date", "sm", "lai", "sm_lai", "sm_squared"]
    ].head()
)

,date,sm,lai,sm_lai,sm_squared
0,2010-01-01,0.168990,0.476,0.080439,0.028558
186,2010-01-17,0.398111,1.776,0.707046,0.158493
2,2010-01-17,0.156495,0.492,0.076996,0.024491
187,2010-01-25,0.396479,1.784,0.707319,0.157196
3,2010-01-25,0.128202,0.492,0.063075,0.016436


In [4]:
import statsmodels.api as sm

validation_records = []
validation_prediction_frames = []

for fold in folds:
    fitting = fold["fitting"]
    validation = fold["validation"]

    for name, features in feature_sets.items():
        X_fit = sm.add_constant(
            fitting[features], has_constant="add"
        )
        X_val = sm.add_constant(
            validation[features], has_constant="add"
        )

        model = sm.OLS(fitting[target], X_fit).fit()
        predicted = model.predict(X_val[X_fit.columns])

        results = validation[
            ["lon", "lat", "date", target]
        ].copy()

        results["candidate"] = name
        results["fold"] = fold["fold"]
        results["predicted"] = predicted
        results["squared_error"] = (
            results["predicted"] - results[target]
        ) ** 2

        validation_prediction_frames.append(results)

        validation_records.append({
            "candidate": name,
            "fold": fold["fold"],
            "n_fit": len(fitting),
            "n_validation": len(validation),
            "rmse": np.sqrt(results["squared_error"].mean()),
        })

validation_results = pd.DataFrame(validation_records)

validation_predictions = pd.concat(
    validation_prediction_frames, ignore_index=True
)

# Aggregate squared errors across all validation observations,
# then take the square root.
validation_summary = (
    validation_predictions.groupby("candidate")
    .agg(
        n_validation=("squared_error", "size"),
        mse=("squared_error", "mean"),
    )
)

validation_summary["rmse"] = np.sqrt(
    validation_summary["mse"]
)

display(
    validation_results.pivot(
        index="candidate", columns="fold", values="rmse"
    ).round(3)
)

display(
    validation_summary[["n_validation", "rmse"]]
    .sort_values("rmse")
    .round(3)
)

validation_results.to_csv(
    "feature_validation_by_fold.csv", index=False
)
validation_summary.to_csv("feature_validation_summary.csv")

fold,1,2,3
candidate,,,
Original + SM²,2.209,2.097,2.235
Original + SM×LAI,2.025,1.245,2.159
Original four,2.725,1.837,2.248


,n_validation,rmse
candidate,,
Original + SM×LAI,73,1.899
Original + SM²,73,2.189
Original four,73,2.292


SM × LAI is the strongest candidate: it improves validation RMSE in all three folds.

| Candidate | Overall validation RMSE |
|---|---:|
| Original + SM × LAI | **1.899°C** |
| Original + SM² | 2.189°C |
| Original four predictors | 2.292°C |

The interaction reduces overall RMSE by about 17% compared with the original four-predictor model. Its improvement is small in the last fold, so we should avoid claiming it is consistently a large benefit.

SM² performs worse than the original model in fold 2. We’ll therefore select the interaction model for this experiment, based on 2010 validation.



In [5]:
train_features = add_features(train)
test_features = add_features(test)

selected_feature_name = validation_summary["rmse"].idxmin()
selected_features = feature_sets[selected_feature_name]

final_specs = {
    "Original four": predictors,
    "Selected engineered": selected_features,
    "Earlier pooled (LAI+SM+HU)": ["lai", "sm", "hu"],
}

extension_models = {}
extension_prediction_frames = []
extension_records = []

for name, features in final_specs.items():
    X_train = sm.add_constant(
        train_features[features], has_constant="add"
    )
    X_test = sm.add_constant(
        test_features[features], has_constant="add"
    )

    model = sm.OLS(train_features[target], X_train).fit()
    extension_models[name] = model

    result = test_features[
        ["lon", "lat", "date", target]
    ].copy()

    result["model"] = name
    result["predicted"] = model.predict(X_test[X_train.columns])
    result["error"] = result["predicted"] - result[target]
    extension_prediction_frames.append(result)

    for (lon, lat), cell in result.groupby(["lon", "lat"]):
        extension_records.append({
            "model": name,
            "lon": lon,
            "lat": lat,
            "n_test": len(cell),
            "rmse": np.sqrt(np.mean(cell["error"] ** 2)),
            "bias": cell["error"].mean(),
        })

extension_results = pd.DataFrame(extension_records)
extension_predictions = pd.concat(
    extension_prediction_frames, ignore_index=True
)

print("Feature set selected using 2010:", selected_feature_name)

display(
    extension_results.pivot(
        index=["lon", "lat"],
        columns="model",
        values="rmse",
    ).round(3)
)

extension_results.to_csv(
    "linear_extensions_2011.csv", index=False
)
extension_predictions.to_csv(
    "linear_extensions_predictions_2011.csv", index=False
)

Feature set selected using 2010: Original + SM×LAI


,model,Earlier pooled (LAI+SM+HU),Original four,Selected engineered
lon,lat,,,
-9.125,52.125,2.471,2.392,2.200
-4.125,41.375,2.756,2.792,3.242
2.125,45.875,2.407,2.329,1.800


The interaction helps two cells in 2011, but worsens the middle cell.

| Cell | Original four RMSE | With SM × LAI | Change |
|---|---:|---:|---|
| −9.125, 52.125 | 2.392°C | 2.200°C | Improved |
| −4.125, 41.375 | 2.792°C | 3.242°C | Worsened |
| 2.125, 45.875 | 2.329°C | 1.800°C | Improved |

This is a useful finding: better overall validation performance does not guarantee improvement at every location in another year. We keep the selected interaction model and report this mixed outcome without changing features based on 2011.